# Inside a 30B model: how does it handle language?

Hands the memory bank and the evidence engine a ~30-billion-parameter language model and asks how it tells languages apart (`docs/NEURO.md`, "A ~30B model and language").

**In plain words:** the model reads the same sentence in 12 languages. The engine then tries four things:
1. **Read:** can it see which language the model thinks the text is in, from the model's insides alone?
2. **Swap:** if it replaces just the "language part" inside the model, does the model change its answer?
3. **Shared meaning:** once the language part is removed, does a sentence look the same as its translations? (meaning and language stored separately)
4. **Speak:** if it adds "French" inside the model while it continues an English sentence, does it switch to French?

Every claim was written down and committed **before** this run (`research/neuro_ledger.jsonl`), after looking only at a tiny model of the same family (Qwen3-0.6B). This notebook runs the test once and records the result, pass or fail.

**Why not Ollama directly?** Ollama only returns text. Reading and editing a model's insides needs its open weights in PyTorch, so this loads the same model from Hugging Face. Section 7 (optional) checks the Ollama build answers the same way.

**Runtime → Change runtime type:**
- **H100, or A100 80 GB** → `qwen3:30b` (Qwen3-30B-A3B, full precision, about 61 GB to download).
- **A100 40 GB or L4** → `qwen3:32b` (Qwen3-32B, shrunk to 4 bits, about 65 GB to download, about 20 GB on the GPU).

The download takes 10–20 minutes. The run takes roughly 1–2 hours: it steers the model through about 10,000 short continuations.

## 1. GPU and model

In [ ]:
import subprocess, shutil, os, json
mem_gb = 0
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
    mem_gb = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                                capture_output=True, text=True).stdout.split()[0]) / 1024
# The model's experts are stored as fused tensors, which 4-bit loading does not shrink: qwen3:30b needs the full ~61 GB.
MODEL = "qwen3:30b" if mem_gb >= 75 else "qwen3:32b" if mem_gb >= 22 else None
print(f"GPU memory {mem_gb:.0f} GB -> {MODEL or 'too small: pick an H100, A100 or L4 runtime'}")
assert MODEL, "this notebook needs a GPU with at least 22 GB"

## 2. Drive, for the results

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/money/neuro"
os.makedirs(SAVE, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"          # weights go on the fast local disk, not Drive
os.environ["MONEY_HOME"] = "/content/var"

## 3. The code

In [ ]:
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
tok = secret("GITHUB_TOKEN")
url = f"https://{tok}@github.com/nikku03/money" if tok else "https://github.com/nikku03/money"
if not os.path.exists("/content/money"):
    !git clone -q -b claude/practical-volta-mbc1b5 {url} /content/money
else:
    !git -C /content/money pull -q
%cd /content/money
!git log --oneline -1
!pip install -q -e ".[lm]" accelerate bitsandbytes

## 4. What was declared for this model
The claims, their thresholds and the pinned model commit, exactly as committed before any result existed.

In [ ]:
recs = [json.loads(l) for l in open("research/neuro_ledger.jsonl")]
mine = [r for r in recs if r["op"] == "declare" and r.get("kind") == "neuro-lang" and r["spec"]["ollama"] == MODEL]
assert mine, f"nothing is declared for {MODEL}: the claims must be declared and committed before they run"
for r in mine:
    s = r["spec"]
    print(f"{r['id']}: threshold {s['threshold']}, {s['model']} @ {s['sha'][:10]}, quant {s['quant']}")
    print("   ", s["claim"][:160], "...")
done = {r["id"] for r in recs if r["op"] == "result"}
print("already has a result:", [r["id"] for r in mine if r["id"] in done] or "none")

## 5. Download the weights (pinned to the declared commit)

In [ ]:
from huggingface_hub import snapshot_download
spec = mine[0]["spec"]
print(snapshot_download(spec["model"], revision=spec["sha"]))

## 6. Run the test
Runs in its own process, so the GPU is free afterwards. Writes the result to `research/neuro_ledger.jsonl` and `research/NEURO.md`, and what it found to the neuro bank.

In [ ]:
!python -m brain neuro lang-test --model {MODEL} 2>&1 | grep -v "Loading weights"

In [ ]:
from IPython.display import Markdown
md_ = open("research/NEURO.md").read()
display(Markdown(md_[md_.find("# How a ~30B model handles language"):]))

## 7. Save

In [ ]:
!cp research/neuro_ledger.jsonl research/NEURO.md {SAVE}/
!tar czf {SAVE}/neuro_bank.tgz -C /content/var neuro/bank
!git status --short research/
print("saved to", SAVE)
# To push the result to the branch (needs GITHUB_TOKEN with write access):
# !git -c user.name="Colab" -c user.email="colab@example.com" commit -am "Language claims N3: {MODEL} results" && git push origin claude/practical-volta-mbc1b5

## 8. Optional: does the Ollama build answer the same way?
Ollama serves a 4-bit GGUF build of the same model. This asks it to name the language of a few TEST sentences, to compare with the run above. Behaviour only: Ollama cannot show the insides. Run it after section 6 so the GPU is free.

In [ ]:
RUN_OLLAMA = False
if RUN_OLLAMA:
    !curl -fsSL https://ollama.com/install.sh | sh
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    import time; time.sleep(5)
    !ollama pull {MODEL}

In [ ]:
if RUN_OLLAMA:
    import urllib.request
    from brain.neuro import lang as G
    P = G.passages(150)
    ok = n = 0
    for code_, name in G.LANGS:
        for s in P[code_][140:150]:
            body = json.dumps({"model": MODEL, "prompt": G.PREFIX + s + G.SUFFIX, "raw": True, "stream": False,
                               "options": {"temperature": 0, "num_predict": 3}}).encode()
            out = json.load(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11434/api/generate", body,
                                                                          {"Content-Type": "application/json"})))["response"]
            ok += out.strip().startswith(name); n += 1
    print(f"Ollama {MODEL} names the language of {ok}/{n} TEST sentences")